# KdV soliton collision

*waves · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/waves/kdv_solitons)

Two solitary waves of the Korteweg–de Vries equation. The taller one is faster, catches up, and both come out of the collision with their shapes intact, only shifted. Derivatives are taken in Fourier space ($\partial_x \to ik$), the stiff $u_{xxx}$ term is solved exactly with an integrating factor and the rest with RK4.

**Vector form**

$$u_t + 6\,u\,u_x + u_{xxx} = 0,\qquad u = \frac{c}{2}\,\mathrm{sech}^2\!\Bigl(\frac{\sqrt c}{2}\,(x - ct)\Bigr)$$

**Index (tensor) form**

$$\partial_t \hat u_k = i k^3\,\hat u_k - 3ik\,F_{kj}\,(u_j u_j),\qquad \hat v_k = e^{-ik^3 t}\,\hat u_k,\qquad M = \sum_j u_j\,\Delta x,\quad P = \sum_j u_j u_j\,\Delta x$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $u_j$ | `u` | (J,) | wave height on a periodic grid (float64) |
| $\hat u_k$ | `torch.fft.fft(u)` | (K,) | Fourier coefficients; $\partial_x \to ik$, $\partial_x^3 \to -ik^3$ |
| $F_{kj}(u_ju_j)$ | `fft(ifft(u_hat).real ** 2) * dealias` | (K,) | nonlinear term $(u^2)$ computed in real space, top third of modes removed (2/3 rule) |
| $e^{ik^3\Delta t}$ | `exp(1j * k**3 * dt)` | (K,) | exact solution of the dispersive part over one step (integrating factor) |
| $M,\ P$ | `mass(u, dx), momentum(u, dx)` | scalar | conserved: $\int u\,dx$ and $\int u^2\,dx$ |

Convention: $u_t + 6uu_x + u_{xxx} = 0$, whose soliton of speed $c$ has height $c/2$, i.e. it travels at twice its amplitude. Taller is faster and narrower. After the collision the tall soliton is ahead of where it would be without interaction and the short one behind; with $\kappa = \sqrt c/2$ the exact shifts are $\frac{1}{\kappa_1}\ln\frac{\kappa_1+\kappa_2}{\kappa_1-\kappa_2}$ forward and $\frac{1}{\kappa_2}\ln\frac{\kappa_1+\kappa_2}{\kappa_1-\kappa_2}$ backward. Dotted: where the solitons would be if they ignored each other. The box is periodic (length 100), so a long run wraps around. The FFT turns every derivative into a multiplication, and the integrating factor removes the severe time-step limit of $u_{xxx}$, leaving only the milder one of the nonlinear term.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/waves/kdv_solitons/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    c1, c2, t_max = params["c_tall"], params["c_short"], params["t_max"]
    x, k, dx = tensors.grid(1024, dev)
    dealias = tensors.dealias_mask(k)
    u0 = tensors.two_solitons(x, c1, c2)

    times, u = solve.solve(u0, k, dealias, t_max)

    # "ghosts": the two solitons translated at their own speeds as if they did not interact (periodic box)
    ghosts = torch.stack(
        [
            equation.soliton(x, c1, (tensors.X_TALL + c1 * t) % tensors.BOX)
            + equation.soliton(x, c2, (tensors.X_SHORT + c2 * t) % tensors.BOX)
            for t in times.tolist()
        ]
    )

    M, P = equation.mass(u, dx), equation.momentum(u, dx)
    print(f"mass     M = {M[0]:.6f}, largest relative change {((M - M[0]).abs().max() / M[0]):.1e}")
    print(f"momentum P = {P[0]:.6f}, largest relative change {((P - P[0]).abs().max() / P[0]):.1e}")
    x1 = solve.peak_position(x, u[-1])
    x2 = solve.second_peak_position(x, u[-1], x1)
    box = tensors.BOX
    shift1 = (x1 - tensors.X_TALL - c1 * t_max + box / 2) % box - box / 2
    shift2 = (x2 - tensors.X_SHORT - c2 * t_max + box / 2) % box - box / 2
    d1, d2 = equation.phase_shifts(c1, c2)
    print(f"tall soliton shifted {shift1:+.3f}  (theory {d1:+.3f} once the collision is over)")
    print(f"short soliton shifted {shift2:+.3f}  (theory {-d2:+.3f})")
    return plot.animate(times, x, u, ghosts, out_dir / "kdv_solitons.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/waves/kdv_solitons/equation.py
r"""Physics → tensor translation.

The Korteweg–de Vries equation u_t + 6 u u_x + u_xxx = 0, with 6 u u_x = 3 (u²)_x, in Fourier space
(∂_x → ik, so ∂_x³ → −ik³):

    \partial_t \hat u_k = i k^3 \hat u_k - 3 i k \, F_{kj} (u_j u_j)

(F_{kj} is the discrete Fourier transform, j summed.) The first term is linear and stiff; it is solved exactly
with the integrating factor \hat v_k = e^{-ik^3 t} \hat u_k, and RK4 handles the nonlinear term.

Single soliton of speed c (height c/2):   u = (c/2) sech²(√c/2 (x − c t)).
Conserved: mass M = ∫ u dx and momentum P = ∫ u² dx.
"""

import math

import torch


def soliton(x: torch.Tensor, c: float, x0: float) -> torch.Tensor:
    """u = (c/2) sech²(√c/2 (x − x0)).   (J,) → (J,)"""
    return 0.5 * c / torch.cosh(0.5 * math.sqrt(c) * (x - x0)) ** 2


def nonlinear(u_hat: torch.Tensor, k: torch.Tensor, dealias: torch.Tensor) -> torch.Tensor:
    r"""−3ik F[u²] with the 2/3 rule.   u_hat, k, dealias: (K,) → (K,)"""
    u = torch.fft.ifft(u_hat).real
    return -3j * k * torch.fft.fft(u * u) * dealias


def if_rk4_step(u_hat: torch.Tensor, k: torch.Tensor, dealias: torch.Tensor, dt: float) -> torch.Tensor:
    r"""One integrating-factor RK4 step: the dispersion e^{ik³Δt} is applied exactly, RK4 does the rest."""
    E = torch.exp(0.5j * k**3 * dt)  # half-step propagator of the linear part
    E2 = E * E
    a = dt * nonlinear(u_hat, k, dealias)
    b = dt * nonlinear(E * (u_hat + a / 2), k, dealias)
    c = dt * nonlinear(E * u_hat + b / 2, k, dealias)
    d = dt * nonlinear(E2 * u_hat + E * c, k, dealias)
    return E2 * u_hat + (E2 * a + 2 * E * (b + c) + d) / 6


def mass(u: torch.Tensor, dx: float) -> torch.Tensor:
    """M = Σ_j u_j Δx.   (…, J) → (…)"""
    return u.sum(dim=-1) * dx


def momentum(u: torch.Tensor, dx: float) -> torch.Tensor:
    """P = Σ_j u_j u_j Δx (j summed).   (…, J) → (…)"""
    return torch.einsum("...j,...j->...", u, u) * dx


def phase_shifts(c1: float, c2: float) -> tuple[float, float]:
    """Exact collision shifts (forward for the faster, backward for the slower), κ = √c / 2."""
    k1, k2 = math.sqrt(c1) / 2, math.sqrt(c2) / 2
    log = math.log((k1 + k2) / abs(k1 - k2))
    return log / k1, log / k2


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/waves/kdv_solitons/tensors.py
"""Build the concrete tensors: periodic grid, wavenumbers, the 2/3 de-aliasing mask and two solitons."""

import math

import torch

from .equation import soliton

BOX = 100.0
X_TALL, X_SHORT = 15.0, 30.0  # starting positions


def grid(n: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor, float]:
    """x (J,) on [0, BOX), wavenumbers k (K = J,), and Δx. float64 keeps the conserved quantities clean."""
    dx = BOX / n
    x = torch.arange(n, device=device, dtype=torch.float64) * dx
    k = 2 * math.pi * torch.fft.fftfreq(n, d=dx).to(device=device, dtype=torch.float64)
    return x, k, dx


def dealias_mask(k: torch.Tensor) -> torch.Tensor:
    """1 for the lowest two thirds of the wavenumbers, 0 above (removes aliasing from u²): (K,)."""
    return (k.abs() < (2 / 3) * k.abs().max()).to(k.dtype)


def two_solitons(x: torch.Tensor, c_tall: float, c_short: float) -> torch.Tensor:
    """u0 (J,): the tall fast soliton behind the short slow one."""
    return soliton(x, c_tall, X_TALL) + soliton(x, c_short, X_SHORT)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/waves/kdv_solitons/solve.py
"""Time-march KdV in Fourier space and locate the solitons afterwards."""

import torch

from problems._common.progress import steps

from .equation import if_rk4_step


def solve(
    u0: torch.Tensor, k: torch.Tensor, dealias: torch.Tensor, t_max: float, n_frames: int = 241
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (F,), u (F, J)) saved at n_frames evenly spaced times."""
    dt = time_step(u0, k)
    n_steps = int(round(t_max / dt))
    save_every = max(1, n_steps // (n_frames - 1))
    u_hat = torch.fft.fft(u0)
    frames, times = [u0], [0.0]
    for step in steps(n_steps, "IF-RK4 steps"):
        u_hat = if_rk4_step(u_hat, k, dealias, dt)
        if (step + 1) % save_every == 0:
            frames.append(torch.fft.ifft(u_hat).real)
            times.append((step + 1) * dt)
    return torch.tensor(times, dtype=u0.dtype, device=u0.device), torch.stack(frames)


def time_step(u0: torch.Tensor, k: torch.Tensor, safety: float = 2.0) -> float:
    """Δt from the nonlinear term only (6 u k Δt ≲ 2.8 for RK4); the dispersion is handled exactly."""
    return min(0.005, safety / (6 * u0.abs().max().item() * k.abs().max().item()))


def peak_position(x: torch.Tensor, u: torch.Tensor) -> float:
    """Sub-grid position of the highest point: fit a parabola through the top three samples."""
    j = int(torch.argmax(u))
    n, dx = len(u), (x[1] - x[0]).item()
    left, mid, right = u[(j - 1) % n].item(), u[j].item(), u[(j + 1) % n].item()
    offset = 0.5 * (left - right) / (left - 2 * mid + right)
    return x[j].item() + offset * dx


def second_peak_position(x: torch.Tensor, u: torch.Tensor, first: float, exclude: float = 6.0) -> float:
    """Position of the highest point farther than `exclude` from `first` (periodic distance)."""
    box = len(u) * (x[1] - x[0]).item()
    dist = torch.remainder(x - first + box / 2, box) - box / 2
    return peak_position(x, torch.where(dist.abs() > exclude, u, torch.zeros_like(u)))


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/waves/kdv_solitons/plot.py
"""Top: the wave now, with dotted 'ghost' solitons that ignore each other. Bottom: the x–t diagram filling in."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(times: torch.Tensor, x: torch.Tensor, u: torch.Tensor, ghosts: torch.Tensor, path: Path) -> Path:
    t, xs = times.cpu().numpy(), x.cpu().numpy()
    U, G = u.cpu().numpy(), ghosts.cpu().numpy()  # (F, J) each
    box = xs[-1] + (xs[1] - xs[0])

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 4.8), gridspec_kw={"height_ratios": [1, 1.3]})
    ghost = ax1.plot(xs, G[0], ":", color="#9ca3af", lw=1.5, label="no interaction")[0]
    line = ax1.plot(xs, U[0], color="#2563eb", lw=2, label="KdV")[0]
    ax1.set_xlim(0, box)
    ax1.set_ylim(-0.1 * U.max(), 1.2 * U.max())
    ax1.set_ylabel("u")
    ax1.legend(loc="upper right", fontsize=8)
    title = ax1.set_title("KdV two-soliton collision   t = 0.0")

    shown = np.full_like(U, np.nan)
    image = ax2.imshow(
        shown, origin="lower", aspect="auto", extent=(0, box, t[0], t[-1]), cmap=plt.get_cmap("Blues", 12),
        vmin=0, vmax=0.7 * U.max(), interpolation="nearest",
    )  # fmt: skip
    ax2.set_xlabel("x")
    ax2.set_ylabel("t")
    ax2.grid(False)

    def update(step: int):
        line.set_ydata(U[step])
        ghost.set_ydata(G[step])
        shown[: step + 1] = U[: step + 1]
        image.set_data(shown)
        title.set_text(f"KdV two-soliton collision   t = {t[step]:.1f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 90), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'c_tall': 4.0,
    'c_short': 1.0,
    't_max': 12.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.waves.kdv_solitons")]:
    del sys.modules[name]
main = importlib.import_module("problems.waves.kdv_solitons.main")
problem_dir = Path("problems/waves/kdv_solitons")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
